<a href="https://colab.research.google.com/github/Elwing-Chou/tibame20260423/blob/main/tibame20260509.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pygame as pg
import random

#pygame初始化
pg.init()

# --- 字型 ---
def get_font(size):
    for f in ['microsoftjhenghei', 'simhei', 'stheitirelight']:
        if f in pg.font.get_fonts():
            return pg.font.SysFont(f, size)
    return pg.font.SysFont(None, size)


# GAME UI
FONT_UI = get_font(32)
col, row = 15, 15
inter = 40
width, height = (col + 2) * inter, (row + 2) * inter
# 產生視窗
screen = pg.display.set_mode([width, height])
# 設定遊戲標題
pg.display.set_caption("踩地雷")

# 遊戲的邏輯
BOMB_COUNT = 20
NOT_CLICKED = -1
# 產生20個不重複的位置當作我的地雷
BOMB = -99
game_board = [[NOT_CLICKED] * col for i in range(row)]
bombs = set()
while True:
    i = random.randint(0, row-1)
    j = random.randint(0, col-1)
    bombs.add((i, j))
    game_board[i][j] = BOMB
    if len(bombs) == BOMB_COUNT:
        break
# 我順便儲存每個位置的中心座標
game_board_centers = [[None] * col for i in range(row)]
# 如果妳是跟雙層list結合 i, j 剛好就是你的位置
for i in range(row):
    for j in range(col):
        x, y = (j + 1) * inter + inter / 2, (i + 1) * inter + inter / 2
        game_board_centers[i][j] = (x, y)
print(game_board_centers)

def draw():
    # 準備第一個圖層
    bg = pg.Surface(screen.get_size())
    # 把畫布填滿某個顏色
    bg.fill([199, 167, 82])

    # test: 把某個地方先隨手設個1-8的數字(代表附近有幾個地雷)
    # game_board[1][1] = "3"
    for i in range(row):
        for j in range(col):
            n = game_board[i][j]
            if n == BOMB:
                # 2. 產生並且畫雷
                # pygame.draw.rect(畫布, 顏色, [x坐標, y坐標, 寬度, 高度], 線寬)
                x_c, y_c = game_board_centers[i][j]
                x, y = x_c - inter / 2, y_c - inter / 2
                pg.draw.rect(bg, [255, 0, 0], [x, y, inter, inter], 0)
            elif n == NOT_CLICKED:
                pass
            else:
                # 3. 當點擊的時候畫出數字
                x_c, y_c = game_board_centers[i][j]
                x, y = x_c - inter / 2, y_c - inter / 2
                pg.draw.rect(bg, [255, 255, 255], [x, y, inter, inter], 0)
                # 字體(字, 平滑話, 顏色)
                t = FONT_UI.render(str(n), 1, [0, 0, 0])
                # 背景(bg)上面疊上t, 字體(center) -> get_rect -> (左上x, 左上y, 寬, 高)
                bg.blit(t, t.get_rect(center=[x_c, y_c]))

    # 1. 畫棋盤的線
    # pygame.draw.line(畫布, 顏色, (x坐標1, y坐標1), (x坐標2, y坐標2), 線寬)
    for i in range(row+1):
        pg.draw.line(bg,
                     [0, 0, 0],
                     [inter, inter*i+inter],
                     [width-inter, inter*i+inter],
                     1)

    for i in range(col+1):
        pg.draw.line(bg,
                     [0, 0, 0],
                     [inter*i+inter, inter],
                     [inter*i+inter, height-inter],
                     1)

    # 你要把圖層放到上一層
    screen.blit(bg, [0, 0])
    # 對畫面進行更新(才會真的秀出來)
    pg.display.update()

def get_bomb_count(game_board, mini, minj):
    # 在這個mini, minj這位置旁邊的九宮格數有幾顆炸彈
    bomb_count = 0
    for i in [-1, 0, 1]:
        for j in [-1, 0, 1]:
            reali, realj = mini + i, minj + j
            # 檢查九宮格是否超出棋盤
            if 0 <= reali < row and 0 <= realj < col:
                if game_board[reali][realj] == BOMB:
                    bomb_count = bomb_count + 1
    # 就真的把炸彈數目填入(代表已經點過了)
    game_board[mini][minj] = bomb_count


# 建立一個永不結束的迴圈(遊戲才不會結束)
draw()
running = True
while running:
    # 收取你的遊戲任何事件(滑鼠點擊/鍵盤按鈕...)
    for event in pg.event.get():
        # 偵測滑鼠點擊以後放掉的動作
        if event.type == pg.MOUSEBUTTONUP:
            # 1. 需要一個儲存媒介, 告訴我點下去的位置到底是安全還是地雷
            # 2. 需要一個近似方式, 來把我點的位置近似到某一個框框
            # (看我點的位置跟哪個框的中心點最近): 為了方便, 我要順便把每一個框的中心點位置儲存起來
            minv, mini, minj = float("inf"), -1, -1
            xm, ym = pg.mouse.get_pos()
            for i in range(row):
                for j in range(col):
                    # 拿到每一個center
                    xc, yc = game_board_centers[i][j]
                    # 把點的位置和每個中心算距離
                    d = abs(xm - xc) + abs(ym - yc)
                    if d < minv:
                        minv, mini, minj = d, i, j
            print("點:", (xm, ym))
            print("近似中心:", game_board_centers[mini][minj])
            if game_board[mini][minj] == BOMB:
                print("遊戲結束")
                running = False
            elif game_board[mini][minj] == NOT_CLICKED:
                get_bomb_count(game_board, mini, minj)
            draw()
        # 如果收到的事件是按x
        if event.type == pg.QUIT:
            # 迴圈就會變成while False
            running = False

pg.quit()

In [7]:
float("-inf")

-inf

In [2]:
l = [80, 20, 50, 30, 40]
# 如果我要找最小值出現在哪個位置
# 我準備一個紀錄, 只要遇到比我的紀錄還小的值, 我們就更新這個紀錄
# minv=遇到最小值, mini=遇到最小值的位置
minv, mini = float("inf"), -1
for i in range(len(l)):
    v = l[i]
    if v < minv:
        minv, mini = v, i
    print(minv, mini)

80 0
20 1
20 1
20 1
20 1


In [ ]:
import pygame as pg
import random

#pygame初始化
pg.init()

# --- 字型 ---
def get_font(size):
    for f in ['microsoftjhenghei', 'simhei', 'stheitirelight']:
        if f in pg.font.get_fonts():
            return pg.font.SysFont(f, size)
    return pg.font.SysFont(None, size)


# GAME UI
FONT_UI = get_font(32)
col, row = 15, 15
inter = 40
width, height = (col + 2) * inter, (row + 2) * inter
# 產生視窗
screen = pg.display.set_mode([width, height])
# 設定遊戲標題
pg.display.set_caption("踩地雷")

# 遊戲的邏輯
BOMB_COUNT = 20
NOT_CLICKED = -1
# 產生20個不重複的位置當作我的地雷
BOMB = -99
game_board = [[NOT_CLICKED] * col for i in range(row)]
bombs = set()
while True:
    i = random.randint(0, row-1)
    j = random.randint(0, col-1)
    bombs.add((i, j))
    game_board[i][j] = BOMB
    if len(bombs) == BOMB_COUNT:
        break
# 我順便儲存每個位置的中心座標
game_board_centers = [[None] * col for i in range(row)]
# 如果妳是跟雙層list結合 i, j 剛好就是你的位置
for i in range(row):
    for j in range(col):
        x, y = (j + 1) * inter + inter / 2, (i + 1) * inter + inter / 2
        game_board_centers[i][j] = (x, y)
print(game_board_centers)

def draw():
    # 準備第一個圖層
    bg = pg.Surface(screen.get_size())
    # 把畫布填滿某個顏色
    bg.fill([199, 167, 82])

    # test: 把某個地方先隨手設個1-8的數字(代表附近有幾個地雷)
    # game_board[1][1] = "3"
    for i in range(row):
        for j in range(col):
            n = game_board[i][j]
            if n == BOMB:
                # 2. 產生並且畫雷
                # pygame.draw.rect(畫布, 顏色, [x坐標, y坐標, 寬度, 高度], 線寬)
                x_c, y_c = game_board_centers[i][j]
                x, y = x_c - inter / 2, y_c - inter / 2
                pg.draw.rect(bg, [255, 0, 0], [x, y, inter, inter], 0)
            elif n == NOT_CLICKED:
                pass
            else:
                # 3. 當點擊的時候畫出數字
                x_c, y_c = game_board_centers[i][j]
                x, y = x_c - inter / 2, y_c - inter / 2
                pg.draw.rect(bg, [255, 255, 255], [x, y, inter, inter], 0)
                # 字體(字, 平滑話, 顏色)
                t = FONT_UI.render(str(n), 1, [0, 0, 0])
                # 背景(bg)上面疊上t, 字體(center) -> get_rect -> (左上x, 左上y, 寬, 高)
                bg.blit(t, t.get_rect(center=[x_c, y_c]))

    # 1. 畫棋盤的線
    # pygame.draw.line(畫布, 顏色, (x坐標1, y坐標1), (x坐標2, y坐標2), 線寬)
    for i in range(row+1):
        pg.draw.line(bg,
                     [0, 0, 0],
                     [inter, inter*i+inter],
                     [width-inter, inter*i+inter],
                     1)

    for i in range(col+1):
        pg.draw.line(bg,
                     [0, 0, 0],
                     [inter*i+inter, inter],
                     [inter*i+inter, height-inter],
                     1)

    # 你要把圖層放到上一層
    screen.blit(bg, [0, 0])
    # 對畫面進行更新(才會真的秀出來)
    pg.display.update()

def get_bomb_count(game_board, mini, minj):
    # 在這個mini, minj這位置旁邊的九宮格數有幾顆炸彈
    bomb_count = 0
    for i in [-1, 0, 1]:
        for j in [-1, 0, 1]:
            reali, realj = mini + i, minj + j
            # 檢查九宮格是否超出棋盤
            if 0 <= reali < row and 0 <= realj < col:
                if game_board[reali][realj] == BOMB:
                    bomb_count = bomb_count + 1
    # 就真的把炸彈數目填入(代表已經點過了)
    game_board[mini][minj] = bomb_count
    # [看看就好] 如果他是0的話要繼續展開
    if bomb_count == 0:
        # 左邊一格的展開
        ni, nj = mini, minj - 1
        if 0 <= ni < row and 0 <= nj < col and game_board[ni][nj] == NOT_CLICKED:
            get_bomb_count(game_board, ni, nj)
        ni, nj = mini, minj + 1
        if 0 <= ni < row and 0 <= nj < col and game_board[ni][nj] == NOT_CLICKED:
            get_bomb_count(game_board, ni, nj)
        ni, nj = mini - 1, minj
        if 0 <= ni < row and 0 <= nj < col and game_board[ni][nj] == NOT_CLICKED:
            get_bomb_count(game_board, ni, nj)
        ni, nj = mini + 1, minj
        if 0 <= ni < row and 0 <= nj < col and game_board[ni][nj] == NOT_CLICKED:
            get_bomb_count(game_board, ni, nj)

# 幫你捕的 勝利條件
def check_win(game_board):
    for i in range(row):
        for j in range(col):
            # 只要有還沒被點到的就還沒贏
            if game_board[i][j] == NOT_CLICKED:
                return False
    return True

# 建立一個永不結束的迴圈(遊戲才不會結束)
draw()
running = True
while running:
    # 收取你的遊戲任何事件(滑鼠點擊/鍵盤按鈕...)
    for event in pg.event.get():
        # 偵測滑鼠點擊以後放掉的動作
        if event.type == pg.MOUSEBUTTONUP:
            # 1. 需要一個儲存媒介, 告訴我點下去的位置到底是安全還是地雷
            # 2. 需要一個近似方式, 來把我點的位置近似到某一個框框
            # (看我點的位置跟哪個框的中心點最近): 為了方便, 我要順便把每一個框的中心點位置儲存起來
            minv, mini, minj = float("inf"), -1, -1
            xm, ym = pg.mouse.get_pos()
            for i in range(row):
                for j in range(col):
                    # 拿到每一個center
                    xc, yc = game_board_centers[i][j]
                    # 把點的位置和每個中心算距離
                    d = abs(xm - xc) + abs(ym - yc)
                    if d < minv:
                        minv, mini, minj = d, i, j
            print("點:", (xm, ym))
            print("近似中心:", game_board_centers[mini][minj])
            if game_board[mini][minj] == BOMB:
                print("遊戲結束")
                running = False
            elif game_board[mini][minj] == NOT_CLICKED:
                get_bomb_count(game_board, mini, minj)
            if check_win(game_board):
                print("贏了!!!!")
                # running = False
            draw()
        # 如果收到的事件是按x
        if event.type == pg.QUIT:
            # 迴圈就會變成while False
            running = False

pg.quit()

In [11]:
import json
import urllib.request as req

url = "https://api.gamer.com.tw/anime/v1/danmu.php?videoSn=48589&geo=TW%2CHK"
f = req.urlopen(url)
content = f.read()
# print(type(content))
content = json.loads(content)
# print(type(content))
danmu = content["data"]["danmu"]
print(danmu)

[{'text': '遠峰不夠變態我可不看喔', 'color': '#FFFFFF', 'size': 1, 'position': 0, 'time': 1, 'sn': 48753101, 'userid': 'stephen91412'}, {'text': '簽', 'color': '#FFFFFF', 'size': 1, 'position': 0, 'time': 27, 'sn': 48753885, 'userid': 'wdps103113'}, {'text': '終於啊~', 'color': '#FFFFFF', 'size': 1, 'position': 0, 'time': 27, 'sn': 48763818, 'userid': 'azovrekar'}, {'text': '簽', 'color': '#FFFFFF', 'size': 1, 'position': 0, 'time': 35, 'sn': 48751106, 'userid': 'maidaz'}, {'text': '竟然動畫化了，希望搞笑的部分可以夠無俚頭', 'color': '#FFFFFF', 'size': 1, 'position': 0, 'time': 49, 'sn': 48753226, 'userid': 'rgps6617'}, {'text': '真人版好看！', 'color': '#FFFFFF', 'size': 1, 'position': 0, 'time': 70, 'sn': 48750530, 'userid': 'qw22956755'}, {'text': '簽', 'color': '#FFFFFF', 'size': 1, 'position': 0, 'time': 70, 'sn': 48752448, 'userid': 'catfishstyle'}, {'text': '沒想到看到動畫化，已經過去18年', 'color': '#FFFFFF', 'size': 1, 'position': 0, 'time': 119, 'sn': 48752989, 'userid': 'ken08520'}, {'text': '簽2026 04 10', 'color': '#FFFFFF', 'siz